# Red-Black Tree

**Difficulty:** Hard &nbsp;|&nbsp; **Topics:** tree, bst, design, rotations
&nbsp;|&nbsp; not on LeetCode - this is the structure inside C++ `std::map`,
Java's `TreeMap`, and half the Linux kernel

You built this once in C. Build it again in Python, from memory first, and let
a referee check every rule after every operation.

A red-black tree is a binary search tree in which every node carries one extra
bit - a color - and five rules hold at all times:

1. Every node is **red** or **black**.
2. The **root** is black.
3. Every leaf (`None` / NIL) counts as **black**.
4. A **red node never has a red child**.
5. Every path from a node down to its leaves passes through the **same number
   of black nodes**.

That's all. Rules 4 and 5 together force the height to stay at most
`2 * log2(n + 1)`, so `insert`, `delete`, and `search` are all `O(log n)` -
no matter how adversarial the input order is.

Implement the `RedBlackTree` class:

- `RedBlackTree()` initializes an empty tree.
- `insert(key)` inserts `key`, keeping all five rules. Duplicates are ignored.
- `search(key)` returns `True` if `key` is in the tree.
- `delete(key)` removes `key`, keeping all five rules. Absent keys are a no-op.

**Contract with the harness:** the tree exposes `self.root`; every node exposes
`.key`, `.left`, `.right`, and `.color` (the strings `"R"` / `"B"`). A leaf may
be `None` *or* a sentinel node whose `key` is `None` - the harness treats both
as a black leaf. Parent pointers are your business; the referee never reads them.

---

### Example

```
insert(10)   ->        10B                (a new root is recolored black)
insert(20)   ->        10B
                          \
                          20R             (new nodes arrive red)
insert(30)   ->        10B
                          \
                          20R
                             \
                             30R          red 20 has red child 30 - rule 4 broken!
             left-rotate(10), recolor:
                       20B
                      /   \
                    10R   30R             all five rules hold again
```

That one step - *rotate, recolor* - is the whole algorithm. Everything else is
case analysis about **where** the red-red pair sits and what color the uncle is.


## Before you write anything

You remember rotations from your C version. Before touching the keyboard, get
the *why* back, not just the moves. Paper first.

**1.** Why balance at all? Insert `1, 2, 3, ..., 1000` in order into a plain
BST - what shape do you get, and what is its height? You met that shape in
#104: a stick. Now every later `search` walks the whole thing. Multiply out
`10^4` ordered inserts on a plain BST - you land in the tens of millions again,
and that number is why this structure exists.

**2.** Write the five rules from memory, then check them against the list
above. Which rules, *together*, forbid the stick? (Hint: a stick of `n` nodes
has one path of length `n` and rule 5 says every path has the same black
count - so the longest path is at most twice the shortest. Say why the factor
is exactly 2, and which rule the factor comes from.)

**3.** The rotation. Draw node `x` with right child `y`, and the three
subtrees `a` (left of `x`), `b` (between), `c` (right of `y`). Write the
inorder walk before a left rotation and after. Same sequence? **That is the
entire trick of this data structure: a rotation changes the shape without
changing the sorted order.** Count the pointers that move - in #206 you moved
one pointer per step; a rotation is three (six with parent pointers). Still
`O(1)`.

**4.** Insert fixup. A new node arrives **red**. Why red and not black?
(Inserting black breaks rule 5 on one whole path - a *global* problem.
Inserting red can only break rule 4 *locally*, right where you stand.) Now the
case split, and this is what you half-remember from C:

- **Uncle is red** -> no rotation at all: recolor parent, uncle, grandparent,
  and the problem moves two levels up. Loop.
- **Uncle is black, zig-zig** (left-left or right-right) -> one rotation at the
  grandparent, swap two colors, done.
- **Uncle is black, zig-zag** (left-right or right-left) -> rotate the parent
  first to straighten the zig-zag, then it *is* the previous case. This
  two-step is the "left-right rotation" you remember - one name, two plain
  rotations.

(Your memory of "left rotation / left-right rotation" as the primary tool is
actually **AVL** vocabulary - AVL rebalances with rotations alone. Red-black
adds the recolor-only case, which is why it does *fewer* rotations per insert
on average. Worth knowing which tree you're quoting.)

**5.** Deletion's ghost. Deleting a red node breaks nothing - check each rule
and say why. Deleting a **black** node steals one black from every path through
it - rule 5 breaks, and the patch is the "double black" you push up or resolve
with the four sibling cases. This is the hard half of the problem, and it is
where the NIL sentinel from your C version earns its keep (question: what
breaks in the fixup loop if a leaf is a bare `None` instead of a node you can
point at and recolor?).

**6.** The referee below re-checks *all five rules plus sortedness* after
**every single operation** by walking the whole tree - `O(n)` per check. Why is
that exactly right for a test harness and criminal in production code? Where
did you last meet a referee like this? (#379's `replay` walked beside your
class with its own set - same idea, bigger rulebook.)


## Two phases - insert first, delete when insert is green

**A - insert + search** *(write this first, get every INSERT case passing)*
`Node` with `key / color / left / right / parent`. Plain BST insert, color the
new node red, then `_insert_fixup` climbs while the parent is red: red uncle ->
recolor and jump to the grandparent; black uncle -> straighten the zig-zag if
needed, one rotation at the grandparent, swap colors, stop. Last line of
`insert`: paint the root black (cheaper than special-casing rule 2). You need
`_left_rotate` and `_right_rotate` - write one, and write the other by
swapping every `left`/`right` in it, exactly like your C version.

**B - delete** *(the hard half)*
Strongly consider the C trick: one shared **NIL sentinel** - a single black
`Node(None)` used as every leaf and as `root.parent` - so the fixup loop can
sit on a leaf, read its color, even temporarily recolor it, without ever
dereferencing `None`. Then it is CLRS verbatim: `_transplant`, successor =
minimum of the right subtree, and `_delete_fixup` with the four sibling cases
(red sibling; black sibling with two black children; near child red; far child
red) - each mirrored left/right. If you only remember one thing from C, let it
be this: **only removing a *black* node needs any fixup at all.**

Route A of #379 had a one-set shortcut; this problem has none. The rules *are*
the data structure - there is no container in the standard library you can
delegate to (that's `SortedContainers`' job in real life, and `std::map`'s in
your C++ life). What you *can* reuse is discipline from the earlier problems:
rotation is #206's pointer surgery times three, the fixup loop is #155's
"restore the invariant before you return", and the referee is #379's harness
with a bigger rulebook.


In [ ]:
import math
import random


In [10]:
RED, BLACK = "R", "B"


class Node:
    def __init__(self, key, color=RED, left=None, right=None, parent=None):
        self.key = key
        self.color = color
        self.left = left
        self.right = right
        self.parent = None


class RedBlackTree:
    def __init__(self):
        self.NIL = Node(data=None, color="black")  # Sentinel NIL node
        self.root = self.NIL

    def rotate_left(self, x):
        y = x.right
        x.right = y.left
        if y.left != self.NIL:
            y.left.parent = x
        y.parent = x.parent
        if x.parent is None:
            self.root = y
        elif x == x.parent.left:
            x.parent.left = y
        else:
            x.parent.right = y
        y.left = x
        x.parent = y

    def rotate_right(self, y):
        x = y.left
        y.left = x.right
        if x.right != self.NIL:
            x.right.parent = y
        x.parent = y.parent
        if y.parent is None:
            self.root = x
        elif y == y.parent.right:
            y.parent.right = x
        else:
            y.parent.left = x
        x.right = y
        y.parent = x

    def insert(self, data):
        new_node = Node(data=data)
        new_node.left = new_node.right = self.NIL

        parent = None
        current = self.root

        while current != self.NIL:
            parent = current
            if new_node.data < current.data:
                current = current.left
            else:
                current = current.right

        new_node.parent = parent
        if parent is None:
            self.root = new_node
        elif new_node.data < parent.data:
            parent.left = new_node
        else:
            parent.right = new_node

        self.fix_insert(new_node)

    def fix_insert(self, z):
        # Case: Parent is red, needing adjustment
        while z.parent and z.parent.color == "red":
            if z.parent == z.parent.parent.left:
                y = z.parent.parent.right  # Uncle node
                if y.color == "red":
                    # Case 2: Both parent and uncle are red
                    z.parent.color = "black"
                    y.color = "black"
                    z.parent.parent.color = "red"
                    z = z.parent.parent  # Recurse upward
                else:
                    # Case 3: Parent is red, uncle is black, and z is a right child
                    if z == z.parent.right:
                        z = z.parent
                        self.rotate_left(z)  # Left-rotate to correct shape
                    # Case 3: Left-rotation done, recolor and rotate
                    z.parent.color = "black"
                    z.parent.parent.color = "red"
                    self.rotate_right(z.parent.parent)  # Right-rotate to fix violation
            else:
                # Symmetric cases for when z's parent is the right child
                y = z.parent.parent.left
                if y.color == "red":
                    # Case 2: Parent and uncle are both red
                    z.parent.color = "black"
                    y.color = "black"
                    z.parent.parent.color = "red"
                    z = z.parent.parent
                else:
                    # Case 3: Parent is red, uncle is black, and z is a left child
                    if z == z.parent.left:
                        z = z.parent
                        self.rotate_right(z)
                    # Case 3: Recoloring and left-rotation
                    z.parent.color = "black"
                    z.parent.parent.color = "red"
                    self.rotate_left(z.parent.parent)
        # Case 1: Root is always black after insertion fix
        self.root.color = "black"

    def transplant(self, u, v):
        if u.parent is None:
            self.root = v
        elif u == u.parent.left:
            u.parent.left = v
        else:
            u.parent.right = v
        v.parent = u.parent

    def delete(self, data):
        z = self.search(self.root, data)
        if z == self.NIL:
            print("Value not found in the tree.")
            return

        y = z
        y_original_color = y.color
        if z.left == self.NIL:
            x = z.right
            self.transplant(z, z.right)
        elif z.right == self.NIL:
            x = z.left
            self.transplant(z, z.left)
        else:
            y = self.minimum(z.right)
            y_original_color = y.color
            x = y.right
            if y.parent == z:
                x.parent = y
            else:
                self.transplant(y, y.right)
                y.right = z.right
                y.right.parent = y
            self.transplant(z, y)
            y.left = z.left
            y.left.parent = y
            y.color = z.color

        if y_original_color == "black":
            self.fix_delete(x)

    def fix_delete(self, x):
        while x != self.root and x.color == "black":
            if x == x.parent.left:
                w = x.parent.right  # Sibling node
                if w.color == "red":
                    # Case 1: Sibling is red
                    w.color = "black"
                    x.parent.color = "red"
                    self.rotate_left(x.parent)
                    w = x.parent.right
                if w.left.color == "black" and w.right.color == "black":
                    # Case 2: Sibling and its children are black
                    w.color = "red"
                    x = x.parent  # Move up the tree
                else:
                    if w.right.color == "black":
                        # Case 3: Sibling is black, left child is red, right is black
                        w.left.color = "black"
                        w.color = "red"
                        self.rotate_right(w)
                        w = x.parent.right
                    # Case 3: Right child of sibling is red
                    w.color = x.parent.color
                    x.parent.color = "black"
                    w.right.color = "black"
                    self.rotate_left(x.parent)
                    x = self.root
            else:
                # Symmetric cases for when x is the right child
                w = x.parent.left
                if w.color == "red":
                    # Case 1: Sibling is red
                    w.color = "black"
                    x.parent.color = "red"
                    self.rotate_right(x.parent)
                    w = x.parent.left
                if w.right.color == "black" and w.left.color == "black":
                    # Case 2: Sibling and its children are black
                    w.color = "red"
                    x = x.parent
                else:
                    if w.left.color == "black":
                        # Case 3: Sibling is black, right child is red, left is black
                        w.right.color = "black"
                        w.color = "red"
                        self.rotate_left(w)
                        w = x.parent.left
                    # Case 3: Left child of sibling is red
                    w.color = x.parent.color
                    x.parent.color = "black"
                    w.left.color = "black"
                    self.rotate_right(x.parent)
                    x = self.root
        # Ensure the final node is black
        x.color = "black"

    def search(self, node, key):
        if node == self.NIL or key == node.data:
            return node
        if key < node.data:
            return self.search(node.left, key)
        return self.search(node.right, key)

    def minimum(self, node):
        while node.left != self.NIL:
            node = node.left
        return node

    def inorder(self):
        self._inorder(self.root)
        print("\n")

    def _inorder(self, node):
        if node != self.NIL:
            self._inorder(node.left)
            print(f"{node.data} ({node.color})", end=" ")
            self._inorder(node.right)


    def preorder(self):
        self._preorder(self.root)
        print("\n")

    def _preorder(self, node):
        if node != self.NIL:
            print(f"{node.data} ({node.color})", end=" ")
            self._preorder(node.left)
            self._preorder(node.right)

    def postorder(self):
        self._postorder(self.root)

    def _postorder(self, node):
        if node != self.NIL:
            self._postorder(node.left)
            self._postorder(node.right)
            print(f"{node.data} ({node.color})", end=" ")


### The test harness

There is no expected-output list - many legal red-black trees hold the same
keys, just as #379's `get()` could hand out any free number. So the referee
checks *properties*, not pictures:

- after **every** `insert`/`delete`, `validate` re-derives all five rules from
  scratch, plus: the inorder walk must read sorted with no duplicates, the
  contents must match a referee `set` walking beside your tree, and the height
  must respect the `2 * log2(n + 1)` bound - the line a plain BST cannot hold;
- `search` must agree with the referee, call for call;
- on a failure you get the op log, the broken rule, and a sideways picture of
  your tree with colors, root at the left.

`stress` builds long random insert/delete/search sequences; the seed makes
every run identical. Run this cell; don't edit it.


In [11]:
def is_nil(node):
    """A black leaf: None, or a sentinel whose key is None."""
    return node is None or node.key is None


def node_color(node):
    return BLACK if is_nil(node) else node.color


def dump(node, depth=0, lines=None):
    """Sideways picture, root at the left, right subtree on top."""
    if lines is None:
        lines = []
    if not is_nil(node):
        dump(node.right, depth + 1, lines)
        lines.append("       " + "     " * depth + f"{node.key}{node.color}")
        dump(node.left, depth + 1, lines)
    return lines


def validate(tree, expected):
    """Re-derive every rule from scratch. Returns None if legal, else what broke."""
    root = tree.root
    if is_nil(root):
        return None if not expected else f"tree is empty but should hold {sorted(expected)}"
    if node_color(root) != BLACK:
        return "rule 2: the root must be black"

    keys = []

    def walk(n):                      # -> black-height; raises on a broken rule
        if is_nil(n):
            return 1                  # rule 3: nil leaves are black
        if node_color(n) == RED and RED in (node_color(n.left), node_color(n.right)):
            raise ValueError(f"rule 4: red node {n.key} has a red child")
        lb = walk(n.left)
        keys.append(n.key)
        rb = walk(n.right)
        if lb != rb:
            raise ValueError(
                f"rule 5: black-heights differ under {n.key} ({lb} left vs {rb} right)")
        return lb + (1 if node_color(n) == BLACK else 0)

    try:
        walk(root)
    except ValueError as broken:
        return str(broken)

    if any(a >= b for a, b in zip(keys, keys[1:])):
        return f"BST order broken: inorder reads {keys}"
    if set(keys) != set(expected):
        missing = sorted(set(expected) - set(keys))
        extra = sorted(set(keys) - set(expected))
        return f"wrong contents: missing {missing}, unexpected {extra}"

    def height(n):
        return 0 if is_nil(n) else 1 + max(height(n.left), height(n.right))

    bound = 2 * math.log2(len(keys) + 1)
    if height(root) > bound:
        return (f"not balanced: height {height(root)} > 2*log2(n+1) = {bound:.1f} "
                f"for n = {len(keys)}")
    return None


def replay(ops, args):
    """Replay an op sequence, re-checking every rule after every mutation.

    Returns (ok, log). On failure the log ends with the broken rule and a
    picture of the tree.
    """
    tree, expected, log = None, set(), []

    for op, a in zip(ops, args):
        if op == "RedBlackTree":
            tree, expected = RedBlackTree(), set()
            log.append("RedBlackTree()")

        elif op == "insert":
            tree.insert(a[0])
            expected.add(a[0])
            log.append(f"insert({a[0]})")

        elif op == "delete":
            tree.delete(a[0])
            expected.discard(a[0])
            log.append(f"delete({a[0]})")

        elif op == "search":
            got = tree.search(a[0])
            log.append(f"search({a[0]}) -> {got!r}")
            if bool(got) != (a[0] in expected):
                log.append(f"   !! search({a[0]}) must be {a[0] in expected}")
                return False, log
            continue

        if op in ("insert", "delete"):
            broken = validate(tree, expected)
            if broken:
                log.append(f"   !! {broken}")
                log.extend(dump(tree.root))
                return False, log

    return True, log


def stress(calls, key_range, seed, mix=(0.55, 0.30)):
    """Random insert/delete/search ops on keys in range(key_range), refereed."""
    random.seed(seed)
    ops, args = ["RedBlackTree"], [[]]
    for _ in range(calls):
        r = random.random()
        if r < mix[0]:
            ops.append("insert"), args.append([random.randrange(key_range)])
        elif r < mix[0] + mix[1]:
            ops.append("delete"), args.append([random.randrange(key_range)])
        else:
            ops.append("search"), args.append([random.randrange(key_range)])
    return replay(ops, args)


def report(name, ok, log, tail=8):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        for line in log[-tail:]:
            print(f"       {line}")


In [12]:
# tests - phase A first: get every INSERT case green before touching delete
INSERT_CASES = [
    ("the C classic: 10, 20, 30 - your first left rotation",
     ["RedBlackTree", "insert", "insert", "insert", "search", "search"],
     [[],             [10],     [20],     [30],     [20],     [99]]),

    ("the mirror: 30, 20, 10 - one right rotation",
     ["RedBlackTree", "insert", "insert", "insert", "search"],
     [[],             [30],     [20],     [10],     [10]]),

    ("zig-zag 10, 30, 20 - the 'left-right' two-step you remember from C",
     ["RedBlackTree", "insert", "insert", "insert", "search"],
     [[],             [10],     [30],     [20],     [20]]),

    ("zig-zag mirrored: 30, 10, 20",
     ["RedBlackTree", "insert", "insert", "insert", "search"],
     [[],             [30],     [10],     [20],     [20]]),

    ("red uncle: 10, 5, 15, 3 - recoloring only, no rotation",
     ["RedBlackTree", "insert", "insert", "insert", "insert", "search"],
     [[],             [10],     [5],      [15],     [3],      [3]]),

    ("climb 1..15 in order - a plain BST would be a stick of height 15",
     ["RedBlackTree"] + ["insert"] * 15 + ["search", "search"],
     [[]] + [[k] for k in range(1, 16)] + [[8], [16]]),

    ("duplicates are ignored - inserting 5 three times stores it once",
     ["RedBlackTree", "insert", "insert", "insert", "search"],
     [[],             [5],      [5],      [5],      [5]]),
]

# phase B: deletion
DELETE_CASES = [
    ("delete a red leaf - the free case, no fixup needed",
     ["RedBlackTree", "insert", "insert", "insert", "delete", "search", "search"],
     [[],             [10],     [5],      [15],     [5],      [5],      [10]]),

    ("delete a black node with one red child",
     ["RedBlackTree", "insert", "insert", "insert", "insert", "delete", "search"],
     [[],             [10],     [5],      [15],     [3],      [5],      [3]]),

    ("delete a black leaf - meet the double black",
     ["RedBlackTree", "insert", "insert", "insert", "insert", "insert", "delete", "search"],
     [[],             [10],     [5],      [15],     [3],      [7],      [15],     [15]]),

    ("delete a node with two children - successor surgery",
     ["RedBlackTree"] + ["insert"] * 7 + ["delete", "search", "search"],
     [[]] + [[k] for k in [50, 25, 75, 10, 30, 60, 90]] + [[50], [50], [60]]),

    ("drain it: insert 1..10, delete 1..10, tree must end empty",
     ["RedBlackTree"] + ["insert"] * 10 + ["delete"] * 10 + ["search"],
     [[]] + [[k] for k in range(1, 11)] + [[k] for k in range(1, 11)] + [[5]]),

    ("delete keys that are not there - the tree must not flinch",
     ["RedBlackTree", "insert", "delete", "delete", "delete", "search"],
     [[],             [10],     [99],     [10],     [10],     [10]]),
]

for name, ops, args in INSERT_CASES:
    report(name, *replay(ops, args))
for name, ops, args in DELETE_CASES:
    report(name, *replay(ops, args))

# the stick test at scale: 300 ordered inserts, every rule checked every time.
# On a plain BST the height bound snaps at the 7th insert - watch it if you
# want by deleting your fixup.
report("climb 1..300 in order, referee watching every insert",
       *replay(["RedBlackTree"] + ["insert"] * 300,
               [[]] + [[k] for k in range(1, 301)]))

# random sequences - insert-heavy, delete-heavy, and search-heavy mixes
for calls, key_range, seed, mix in [(60, 8, 1, (0.55, 0.30)),
                                    (400, 30, 2, (0.55, 0.30)),
                                    (1500, 200, 3, (0.45, 0.40)),
                                    (3000, 10000, 4, (0.60, 0.25))]:
    report(f"stress: {calls} random ops on keys 0..{key_range - 1} (seed {seed})",
           *stress(calls, key_range, seed, mix))

# see it, do not just trust the pass/fail
print("\nyour tree after inserting 1..10 (root at the left):")
t = RedBlackTree()
for k in range(1, 11):
    t.insert(k)
for line in dump(t.root):
    print(line)


TypeError: Node.__init__() got an unexpected keyword argument 'data'

## After it passes

- **Measure the claim.** Insert `1..10^4` in order and print the height - the
  bound says at most `2 * log2(10^4 + 1) = 26.6`; you'll typically see far less.
  Then try the same on a fixup-free BST and watch the referee stop it at insert
  number 7. The gap between 14-ish and 10,000 is the entire sales pitch.
- **AVL vs red-black - the vocabulary you remembered.** AVL keeps a stricter
  balance (heights of siblings differ by at most 1) and rebalances with
  rotations only - that's where "left rotation, left-right rotation" as *the*
  toolkit comes from. Red-black tolerates a sloppier balance (factor 2) and
  pays for it with cheaper updates - the red-uncle case is a recolor, no
  pointers move. That trade is why `std::map`, Java's `TreeMap`, and the Linux
  kernel chose red-black, while databases that read far more than they write
  often prefer AVL or B-trees. Which would you pick for #379's phone company?
- **Why do new nodes arrive red?** Say it in one sentence about *local* versus
  *global* damage. Then say which rule a freshly-painted-black root breaks if
  you skip the last line of `insert`.
- **The sentinel, revisited.** If you built phase B with `None` leaves, count
  the `if x is not None` guards inside `_delete_fixup`. Rebuild with the shared
  NIL sentinel and count again. Your C version already knew this - one dummy
  node buys away a dozen branches. Where did a dummy/sentinel save you before
  in this repo? (#206's dummy head is the same trick wearing a smaller hat.)
- **The referee is `O(n)` per op - your tree must not be.** `validate` walks
  everything after every mutation; your `insert` must touch only one root-leaf
  path. Prove it to yourself: which loop in your fixup can run more than a
  constant number of times, and what bounds it? (Answer: the red-uncle climb,
  bounded by height, so `O(log n)` - and at most **two** rotations per insert,
  ever. Deletion: at most three.)
- **One structure, two invariants.** #155 and #379-route-B kept two structures
  describing one fact. This tree is the opposite: one structure carrying two
  facts at once - sorted order (BST rule) and balance (color rules) - and every
  operation must restore *both* before returning. Which of the two did the
  rotation protect, and which did the recoloring protect?
- **Siblings:** #98 Validate Binary Search Tree (your `validate`'s inorder
  check *is* the answer); #110 Balanced Binary Tree (the AVL check); #450
  Delete Node in a BST (your phase B minus the colors - do it as a victory
  lap); #701 Insert into a BST. In real Python, reach for `sortedcontainers`
  or `bisect` before hand-rolling this - but now you know what they're hiding.
